In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline


In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA health")

df_v2 = spark.table("gold_mart_patient_day_v2")
df_v2.printSchema()

In [0]:
num_features = [
    "avg_hr",
    "avg_sys",
    "avg_dia",
    "avg_spo2",
    "cholesterol",
    "glucose",
    "hemoglobin",
    "age_est"
]
cat_features = ["gender", "region"]

df_ml = df_v2.select(
    ["patient_id", "date", "risk_label_v2", "risk_score_v2"] + num_features + cat_features
).dropna()

print("Rows df_ml:", df_ml.count())
df_ml.show(5, truncate=False)

In [0]:
# Label indexer
label_indexer = StringIndexer(
    inputCol="risk_label_v2",
    outputCol="label",
    handleInvalid="keep"
)

# Categorical indexers
gender_indexer = StringIndexer(
    inputCol="gender",
    outputCol="gender_idx",
    handleInvalid="keep"
)

region_indexer = StringIndexer(
    inputCol="region",
    outputCol="region_idx",
    handleInvalid="keep"
)

onehot = OneHotEncoder(
    inputCols=["gender_idx", "region_idx"],
    outputCols=["gender_oh", "region_oh"]
)

# Assemble features
assembler_rf = VectorAssembler(
    inputCols=num_features + ["gender_oh", "region_oh"],
    outputCol="features_rf"
)

rf = RandomForestClassifier(
    featuresCol="features_rf",
    labelCol="label",
    numTrees=100,
    maxDepth=8,
    seed=42
)

pipeline_rf = Pipeline(stages=[
    label_indexer,
    gender_indexer,
    region_indexer,
    onehot,
    assembler_rf,
    rf
])

rf_model = pipeline_rf.fit(df_ml)   # train trên toàn bộ df_ml
print("RF model trained.")


In [0]:
pred_full = rf_model.transform(df_ml)

# Lấy labels từ StringIndexerModel (stage 0)
label_indexer_model = rf_model.stages[0]
labels = label_indexer_model.labels
print("Labels order:", labels)

def make_label_udf(labels_list):
    def _map(idx):
        if idx is None:
            return None
        i = int(idx)
        if 0 <= i < len(labels_list):
            return labels_list[i]
        return None
    return F.udf(_map, StringType())

label_udf = make_label_udf(labels)

pred_full = pred_full.withColumn(
    "risk_label_pred",
    label_udf(F.col("prediction"))
)

pred_full.select(
    "risk_label_v2", "prediction", "risk_label_pred"
).show(10, truncate=False)


In [0]:
from pyspark.sql.types import DoubleType

# Xác định index từng label
idx_normal   = labels.index("Normal")   if "Normal"   in labels else None
idx_elevated = labels.index("Elevated") if "Elevated" in labels else None
idx_high     = labels.index("High")     if "High"     in labels else None

print("idx_normal:", idx_normal, " idx_elevated:", idx_elevated, " idx_high:", idx_high)

def make_prob_udf(idx):
    if idx is None:
        return F.udf(lambda v: None, DoubleType())
    def _get(v):
        if v is None:
            return None
        try:
            return float(v[idx])
        except Exception:
            return None
    return F.udf(_get, DoubleType())

prob_normal_udf   = make_prob_udf(idx_normal)
prob_elev_udf     = make_prob_udf(idx_elevated)
prob_high_udf     = make_prob_udf(idx_high)

pred_full = (
    pred_full
        .withColumn("prob_normal",   prob_normal_udf("probability"))
        .withColumn("prob_elevated", prob_elev_udf("probability"))
        .withColumn("prob_high",     prob_high_udf("probability"))
)

pred_full.select(
    "risk_label_v2", "risk_label_pred",
    "prob_normal", "prob_elevated", "prob_high"
).show(10, truncate=False)


In [0]:
serving_df = pred_full.select(
    "patient_id",
    "date",
    "risk_label_v2",          # guideline + noise (ground truth)
    "risk_label_pred",        # dự đoán RF
    "prob_normal",
    "prob_elevated",
    "prob_high",
    "risk_score_v2",          # risk score guideline

    # Features chính cho BI
    "avg_hr",
    "avg_sys",
    "avg_dia",
    "avg_spo2",
    "cholesterol",
    "glucose",
    "hemoglobin",
    "age_est",
    "gender",
    "region"
)

serving_df.show(10, truncate=False)

(
    serving_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("gold_serving_risk_label_v2")
)

print("Saved table: gold_serving_risk_label_v2")


In [0]:
spark.sql("SHOW TABLES").show(truncate=False)
spark.table("gold_serving_risk_label_v2").show(5, truncate=False)
